# Assignment 4.1 — Model Store
**Mostafa Zamaniturk**

Register the Lab 4.1 XGBoost breast-cancer model in SageMaker Model Registry:

1. **Model Group** — versioned container for this use case
2. **Model Package** — this experiment’s image, artifact, and inference spec
3. **Model Card** — qualitative snapshot for future maintainers

**Prereq:** finish Lab 4.1 (`01-train-and-deploy.ipynb`) first so a completed training job exists.

**Kernel:** Python 3 (Data Science) in SageMaker Studio.

**Submission:** screenshot the `describe_*` outputs in Parts 1–3 and compile them into a PDF.

## 0. Setup and reuse Lab 4.1 training artifacts

In [1]:
import os
import json
from time import gmtime, strftime

import boto3
import sagemaker
from sagemaker import get_execution_role

os.environ["SAGEMAKER_SUPPRESS_V2_WARNING"] = "1"

role = get_execution_role()
sess = sagemaker.Session()
region = sess.boto_region_name
sm = boto3.client("sagemaker", region_name=region)

bucket = sess.default_bucket()
prefix = "DEMO-breast-cancer-prediction-xgboost-highlevel"

print("region:", region)
print("bucket:", bucket)
print("role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


region: us-east-1
bucket: sagemaker-us-east-1-682123396235
role: arn:aws:iam::682123396235:role/LabRole


In [2]:
# Reuse the latest completed Lab 4.1 XGBoost training job (name starts with xgb-).
# If you want a specific job, set TRAINING_JOB_NAME = "xgb-YYYY-mm-dd-HH-MM-SS"
TRAINING_JOB_NAME = None

if TRAINING_JOB_NAME is None:
    jobs = sm.list_training_jobs(
        NameContains="xgb-",
        StatusEquals="Completed",
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=10,
    )["TrainingJobSummaries"]
    if not jobs:
        raise RuntimeError(
            "No completed xgb- training job found. Run 01-train-and-deploy.ipynb first, "
            "or set TRAINING_JOB_NAME to that job name."
        )
    TRAINING_JOB_NAME = jobs[0]["TrainingJobName"]

info = sm.describe_training_job(TrainingJobName=TRAINING_JOB_NAME)
model_data_url = info["ModelArtifacts"]["S3ModelArtifacts"]
training_job_arn = info["TrainingJobArn"]
hyperparameters = info.get("HyperParameters", {})

# Same XGBoost image used in Lab 4.1
image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost", region=region, version="1.7-1"
)

final_metrics = {
    m["MetricName"]: m["Value"] for m in info.get("FinalMetricDataList", [])
}

print("training job:", TRAINING_JOB_NAME)
print("training arn:", training_job_arn)
print("model artifact:", model_data_url)
print("image:", image_uri)
print("hyperparameters:", hyperparameters)
print("final metrics:", final_metrics)

training job: xgb-2026-09-26-18-54-32
training arn: arn:aws:sagemaker:us-east-1:682123396235:training-job/xgb-2026-09-26-18-54-32
model artifact: s3://sagemaker-us-east-1-682123396235/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-09-26-18-54-32/xgb-2026-09-26-18-54-32/output/model.tar.gz
image: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
hyperparameters: {'eta': '0.2', 'gamma': '4', 'max_depth': '5', 'min_child_weight': '6', 'num_round': '100', 'objective': 'binary:logistic', 'subsample': '0.8', 'verbosity': '0'}
final metrics: {'validation:logloss': 0.015209999866783619, 'train:logloss': 0.015209999866783619}


## Part 1: Set Up Model Group

The Model Group holds versioned Model Packages. Each time we change the algorithm, features, data, or hyperparameters, we add a new package to this group.

**Name:** `xgboost-breast-cancer-detection`  
**Description:** tracks XGBoost experiments that classify UCI breast-mass images as malignant vs benign.

In [3]:
model_package_group_name = "xgboost-breast-cancer-detection"
model_package_group_description = (
    "XGBoost binary classifiers for UCI Breast Cancer Wisconsin (Diagnostic). "
    "Tracks algorithm, features, hyperparameters, artifacts, and deployments."
)

try:
    create_group_response = sm.create_model_package_group(
        ModelPackageGroupName=model_package_group_name,
        ModelPackageGroupDescription=model_package_group_description,
    )
    print("Created Model Group:")
    print(create_group_response)
except sm.exceptions.ClientError as e:
    if e.response["Error"]["Code"] in ("ValidationException", "ResourceInUse"):
        print("Model Group already exists; reusing it.")
    else:
        raise

Created Model Group:
{'ModelPackageGroupArn': 'arn:aws:sagemaker:us-east-1:682123396235:model-package-group/xgboost-breast-cancer-detection', 'ResponseMetadata': {'RequestId': '69ed6aa7-125e-4b28-b6f7-ac563bd2caf7', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amzn-requestid': '69ed6aa7-125e-4b28-b6f7-ac563bd2caf7', 'strict-transport-security': 'max-age=47304000; includeSubDomains', 'x-frame-options': 'DENY', 'content-security-policy': "frame-ancestors 'none'", 'cache-control': 'no-cache, no-store, must-revalidate', 'x-content-type-options': 'nosniff', 'content-type': 'application/x-amz-json-1.1', 'content-length': '119', 'date': 'Sat, 26 Sep 2026 20:07:59 GMT'}, 'RetryAttempts': 0}}


### Submission screenshot — `describe_model_package_group`

In [4]:
describe_group_response = sm.describe_model_package_group(
    ModelPackageGroupName=model_package_group_name
)
describe_group_response

{'ModelPackageGroupName': 'xgboost-breast-cancer-detection',
 'ModelPackageGroupArn': 'arn:aws:sagemaker:us-east-1:682123396235:model-package-group/xgboost-breast-cancer-detection',
 'ModelPackageGroupDescription': 'XGBoost binary classifiers for UCI Breast Cancer Wisconsin (Diagnostic). Tracks algorithm, features, hyperparameters, artifacts, and deployments.',
 'CreationTime': datetime.datetime(2026, 9, 26, 20, 7, 59, 119000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:682123396235:user-profile/d-t1mregqi78oz/default-1788640020515',
  'UserProfileName': 'default-1788640020515',
  'DomainId': 'd-t1mregqi78oz',
  'IamIdentity': {'Arn': 'arn:aws:sts::682123396235:assumed-role/LabRole/SageMaker',
   'PrincipalId': 'AROAZ5UNZASFYBYEFQ5XM:SageMaker'}},
 'ModelPackageGroupStatus': 'Completed',
 'ResponseMetadata': {'RequestId': 'e254ea4f-e103-4d93-9a83-946cd9b238cf',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'e254ea4f-e103-4d93-9a83-9

## Part 2: Set Up Model Package

The Model Package records this experiment: inference container image, `model.tar.gz` in S3, supported instance types, and I/O formats used for real-time and batch inference in Lab 4.1.

In [5]:
create_package_response = sm.create_model_package(
    ModelPackageGroupName=model_package_group_name,
    ModelPackageDescription=(
        f"Lab 4.1 XGBoost 1.7-1 binary:logistic model from training job "
        f"{TRAINING_JOB_NAME}. CSV in / probability out. Deployed to batch "
        f"transform and a real-time ml.m5.xlarge endpoint."
    ),
    InferenceSpecification={
        "Containers": [
            {
                "Image": image_uri,
                "ModelDataUrl": model_data_url,
            }
        ],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.xlarge"],
        "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
    },
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={
        "training_job": TRAINING_JOB_NAME,
        "algorithm": "xgboost-1.7-1",
        "objective": hyperparameters.get("objective", "binary:logistic"),
        "owner": "Mostafa Zamaniturk",
    },
)

model_package_arn = create_package_response["ModelPackageArn"]
print("Created Model Package:")
print(model_package_arn)

Created Model Package:
arn:aws:sagemaker:us-east-1:682123396235:model-package/xgboost-breast-cancer-detection/1


### Submission screenshot — `describe_model_package`

In [6]:
describe_package_response = sm.describe_model_package(
    ModelPackageName=model_package_arn
)
describe_package_response

{'ModelPackageGroupName': 'xgboost-breast-cancer-detection',
 'ModelPackageVersion': 1,
 'ModelPackageRegistrationType': 'Registered',
 'ModelPackageArn': 'arn:aws:sagemaker:us-east-1:682123396235:model-package/xgboost-breast-cancer-detection/1',
 'ModelPackageDescription': 'Lab 4.1 XGBoost 1.7-1 binary:logistic model from training job xgb-2026-09-26-18-54-32. CSV in / probability out. Deployed to batch transform and a real-time ml.m5.xlarge endpoint.',
 'CreationTime': datetime.datetime(2026, 9, 26, 20, 9, 26, 61000, tzinfo=tzlocal()),
 'InferenceSpecification': {'Containers': [{'Image': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1',
    'ImageDigest': 'sha256:b4f13edb198529c460692015797fa1ca6a8ff1ed64a149297174d922121b8fc4',
    'ModelDataUrl': 's3://sagemaker-us-east-1-682123396235/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-09-26-18-54-32/xgb-2026-09-26-18-54-32/output/model.tar.gz',
    'ModelDataETag': '840491979a7a6fe03d84be964e03acd2

## Part 3: Write the Model Card

The Model Card is a brief snapshot: algorithm, training setup, features, owner, intended use, and evaluation notes so we do not repeat the same experiments.

In [8]:
train_s3 = f"s3://{bucket}/{prefix}/train"
validation_s3 = f"s3://{bucket}/{prefix}/validation"

val_logloss = float(final_metrics.get("validation:logloss", 0.01521))
train_logloss = float(final_metrics.get("train:logloss", 0.01521))

hyper_param_list = [
    {"name": name, "value": str(value)}
    for name, value in hyperparameters.items()
]

model_card_content = {
    "model_overview": {
        "model_description": (
            "Binary XGBoost classifier that estimates the probability a breast "
            "mass is malignant using 30 numeric diagnostic features from the UCI "
            "Breast Cancer Wisconsin (Diagnostic) dataset."
        ),
        "model_creator": "Mostafa Zamaniturk",
        "model_owner": "Mostafa Zamaniturk",
        "algorithm_type": "XGBoost 1.7-1 (SageMaker built-in, binary:logistic)",
        "problem_type": "Binary Classification",
        "model_artifact": [model_data_url],
    },
    "intended_uses": {
        "purpose_of_model": (
            "Educational MLOps demo: document training, batch transform, and "
            "real-time endpoint deployment for a diagnostic classifier."
        ),
        "intended_uses": (
            "Course assignment and lab reproduction. Not for clinical diagnosis."
        ),
        "factors_affecting_model_efficiency": (
            "Small sample (569 rows). Input must be 30 numeric features in the "
            "same order as training, CSV, no header, no id/diagnosis columns."
        ),
        "risk_rating": "High",
        "explanations_for_risk_rating": (
            "Medical diagnosis is high-stakes. This model is a class exercise only."
        ),
    },
    "business_details": {
        "business_problem": (
            "Classify breast-mass images as malignant (1) or benign (0) to "
            "practice SageMaker training, batch inference, and Model Registry."
        ),
        "business_stakeholders": "AAI-540 MLOps course staff and student",
        "line_of_business": "Education / MLOps",
    },
    "training_details": {
        "objective_function": {
            "function": {"function": "Minimize", "facet": "logloss"},
            "notes": "XGBoost objective binary:logistic; 100 boosting rounds.",
        },
        "training_observations": (
            "80/10/10 random split. Dropped id from train/val. Batch set kept id "
            "for join/filter demos and dropped diagnosis to avoid leakage. "
            "Instance: ml.m5.xlarge. Experiments: Lab 4.1 default hyperparameters "
            "only (max_depth=5, eta=0.2, gamma=4, min_child_weight=6, subsample=0.8, "
            "num_round=100). No extra HPO this version."
        ),
        "training_job_details": {
            "training_arn": training_job_arn,
            "training_datasets": [train_s3, validation_s3],
            "training_environment": {"container_image": [image_uri]},
            "training_metrics": [
                {
                    "name": "train:logloss",
                    "value": train_logloss,
                    "notes": "Final training logloss",
                },
                {
                    "name": "validation:logloss",
                    "value": val_logloss,
                    "notes": "Final validation logloss",
                },
            ],
            "hyper_parameters": hyper_param_list,
        },
    },
    "evaluation_details": [
        {
            "name": "lab-4-1-validation-and-serving",
            "evaluation_observation": (
                "Training ran 100 rounds; logloss fell quickly then flattened near "
                f"{val_logloss:.5f}. Batch transform wrote per-row malignant "
                "probabilities (text/csv). Real-time endpoint on ml.m5.xlarge "
                "returned a probability for a single CSV row. Threshold (~0.5) "
                "maps probability to class. No separate holdout accuracy report "
                "in this assignment."
            ),
            "datasets": [validation_s3],
            "metric_groups": [
                {
                    "name": "xgboost-logloss",
                    "metric_data": [
                        {
                            "name": "validation_logloss",
                            "type": "number",
                            "value": val_logloss,
                            "notes": "Final validation logloss from the training job",
                        }
                    ],
                }
            ],
        }
    ],
    "additional_information": {
        "ethical_considerations": (
            "Healthcare data and predictions can affect patients. Do not use this "
            "model for diagnosis or treatment decisions."
        ),
        "caveats_and_recommendations": (
            "Features: radius, texture, perimeter, area, smoothness, compactness, "
            "concavity, concave points, symmetry, fractal dimension (mean, se, "
            "worst). Next experiments to try: scale features, class weighting, "
            "early stopping, and a held-out AUC/F1 report before any production use."
        ),
        "custom_details": {
            "input_features": (
                "30 numeric cytology features; drop id and diagnosis at inference"
            ),
            "serving": "batch transform + real-time endpoint (ml.m5.xlarge)",
            "owner": "Mostafa Zamaniturk",
        },
    },
}

model_card_name = "xgboost-breast-cancer-detection-card"

try:
    create_card_response = sm.create_model_card(
        ModelCardName=model_card_name,
        Content=json.dumps(model_card_content),
        ModelCardStatus="Draft",
    )
    print("Created Model Card:")
    print(create_card_response)
except sm.exceptions.ClientError as e:
    if e.response["Error"]["Code"] in ("ValidationException", "ConflictException", "ResourceInUse"):
        print("Model Card already exists; updating content.")
        sm.update_model_card(
            ModelCardName=model_card_name,
            Content=json.dumps(model_card_content),
            ModelCardStatus="Draft",
        )
    else:
        raise

Model Card already exists; updating content.


### Submission screenshot — `describe_model_card`

In [9]:
describe_card_response = sm.describe_model_card(ModelCardName=model_card_name)
describe_card_response

{'ModelCardArn': 'arn:aws:sagemaker:us-east-1:682123396235:model-card/xgboost-breast-cancer-detection-card',
 'ModelCardName': 'xgboost-breast-cancer-detection-card',
 'ModelCardVersion': 2,
 'Content': '{"model_overview": {"model_description": "Binary XGBoost classifier that estimates the probability a breast mass is malignant using 30 numeric diagnostic features from the UCI Breast Cancer Wisconsin (Diagnostic) dataset.", "model_creator": "Mostafa Zamaniturk", "model_owner": "Mostafa Zamaniturk", "algorithm_type": "XGBoost 1.7-1 (SageMaker built-in, binary:logistic)", "problem_type": "Binary Classification", "model_artifact": ["s3://sagemaker-us-east-1-682123396235/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-09-26-18-54-32/xgb-2026-09-26-18-54-32/output/model.tar.gz"]}, "intended_uses": {"purpose_of_model": "Educational MLOps demo: document training, batch transform, and real-time endpoint deployment for a diagnostic classifier.", "intended_uses": "Course assignme

In [10]:
# Readable view of the card JSON (optional extra screenshot)
print(json.dumps(json.loads(describe_card_response["Content"]), indent=2))

{
  "model_overview": {
    "model_description": "Binary XGBoost classifier that estimates the probability a breast mass is malignant using 30 numeric diagnostic features from the UCI Breast Cancer Wisconsin (Diagnostic) dataset.",
    "model_creator": "Mostafa Zamaniturk",
    "model_owner": "Mostafa Zamaniturk",
    "algorithm_type": "XGBoost 1.7-1 (SageMaker built-in, binary:logistic)",
    "problem_type": "Binary Classification",
    "model_artifact": [
      "s3://sagemaker-us-east-1-682123396235/DEMO-breast-cancer-prediction-xgboost-highlevel/output/xgb-2026-09-26-18-54-32/xgb-2026-09-26-18-54-32/output/model.tar.gz"
    ]
  },
  "intended_uses": {
    "purpose_of_model": "Educational MLOps demo: document training, batch transform, and real-time endpoint deployment for a diagnostic classifier.",
    "intended_uses": "Course assignment and lab reproduction. Not for clinical diagnosis.",
    "factors_affecting_model_efficiency": "Small sample (569 rows). Input must be 30 numeric fe

## Cleanup (optional)

Do **not** delete these until after you take screenshots and export the PDF.

In [ ]:
# sm.delete_model_card(ModelCardName=model_card_name)
# sm.delete_model_package(ModelPackageName=model_package_arn)
# sm.delete_model_package_group(ModelPackageGroupName=model_package_group_name)
# print("Deleted Model Card, Package, and Group.")